In [1]:
#@title 0) setup
%%capture
!pip install transformers accelerate datasets torch lm-eval pandas

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import pandas as pd
from IPython.display import display
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
MODEL_NAMES = {
    "workshop-v1": "JustinAngel/workshop-v1-pretraining",
    "gpt2-small": "gpt2"
}

# 1. Load Models & Hello World

In [ ]:
def load_model_and_tokenizer(model_id):
    tokenizer_id = "gpt2" if "workshop" in model_id else model_id

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

In [ ]:
def generate_hello_world(model, tokenizer, prompt="Hello, world! Today I want to"):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(inputs["input_ids"], max_new_tokens=30, do_sample=False)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAMES["workshop-v1"])
print(generate_hello_world(model, tokenizer))
del model; torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/98 [00:00<?, ?it/s]

Hello, world! Today I want to share my experience with the students. I have been working on a project for a while and I have been working on a project for a while. I


In [ ]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAMES["gpt2-small"])
print(generate_hello_world(model, tokenizer))
del model; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Hello, world! Today I want to talk about the new game, The Legend of Zelda: Breath of the Wild.

The Legend of Zelda: Breath of the Wild is a new


# 2. Manual LAMBADA Evaluation

In [ ]:
import textwrap

lambada_dataset = load_dataset("cimec/lambada", split="test")
print(f"LAMBADA test set: {len(lambada_dataset)} examples")
example_text = lambada_dataset[0]['text']
print(f"\nExample:\n{textwrap.fill(example_text, width=80)}")

LAMBADA test set: 5153 examples

Example:
in my palm is a clear stone , and inside it is a small ivory statuette . a guardian angel . `` figured if you 're going to be out at night getting hit by cars , you might as well have some backup . '' i look at him , feeling stunned . like this is some sort of sign . but as i stare at harlin , his mouth curved in a confident grin , i do n't care about signs


In [ ]:
def split_lambada_example(text):
    words = text.strip().split()
    context = " ".join(words[:-1])
    target_last_word = words[-1]
    return context, target_last_word

def predict_last_word(model, tokenizer, context):
    inputs = tokenizer(context, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(inputs["input_ids"], max_new_tokens=1, do_sample=False)
    predicted_token = output_ids[0, -1]
    return tokenizer.decode(predicted_token).strip()

def evaluate_lambada(model, tokenizer, dataset, num_examples):
    correct = 0
    for i in range(num_examples):
        context, target = split_lambada_example(dataset[i]["text"])
        prediction = predict_last_word(model, tokenizer, context)
        if prediction.lower() == target.lower().rstrip(".,!?;:"):
            correct += 1
    accuracy = correct / num_examples
    return {"correct": correct, "total": num_examples, "accuracy": accuracy}


In [ ]:
NUM_LAMBADA_EXAMPLES = 100

lambada_results = {}

for model_label, model_id in MODEL_NAMES.items():
    print(f"Evaluating {model_label}...")
    model, tokenizer = load_model_and_tokenizer(model_id)
    lambada_results[model_label] = evaluate_lambada(
        model, tokenizer, lambada_dataset, NUM_LAMBADA_EXAMPLES
    )
    del model; torch.cuda.empty_cache()

Evaluating workshop-v1...


Loading weights:   0%|          | 0/98 [00:00<?, ?it/s]

Evaluating gpt2-small...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
lambada_df = pd.DataFrame(lambada_results).T
lambada_df["accuracy"] = lambada_df["accuracy"].map("{:.1%}".format)
lambada_df.index.name = "model"
display(lambada_df)

,correct,total,accuracy
model,,,
workshop-v1,14.0,100.0,14.0%
gpt2-small,19.0,100.0,19.0%


# 3. MMLU via lm-evaluation-harness

In [ ]:
!lm_eval --model hf \
    --model_args pretrained=gpt2 \
    --tasks mmlu \
    --num_fewshot 5 \
    --limit 50 \
    --batch_size auto \
    --device cuda:0

2026-04-11:22:54:55 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-11:22:55:07 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-04-11:22:55:10 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-11:22:55:10 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'gpt2'}
2026-04-11:22:55:17 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-04-11:22:55:19 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 148/148 [00:00<00:00, 672.05it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPE

# 4. Exercise: choose a benchmark and run it

[lm_eval task list](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/lm_eval/tasks/README.md)



In [ ]:
assert False, "call lm_eval to run a benchmark on gpt2"

In [ ]:
#@title answer - lambada
!lm_eval --model hf \
    --model_args pretrained=gpt2 \
    --tasks lambada \
    --limit 50 \
    --device cuda:0

2026-04-11:23:26:16 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-11:23:26:29 INFO     [_cli.run:376] Selected Tasks: ['lambada']
2026-04-11:23:26:33 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-11:23:26:33 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'gpt2'}
2026-04-11:23:26:44 INFO     [models.huggingface:161] Using device 'cuda:0'
config.json: 100% 665/665 [00:00<00:00, 1.98MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 39.6kB/s]
vocab.json: 1.04MB [00:00, 2.16MB/s]
merges.txt: 456kB [00:00, 979kB/s]
tokenizer.json: 1.36MB [00:00, 1.76MB/s]
2026-04-11:23:26:51 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
model.safetensors: 100% 548M/548M [00:03<00:00,

In [3]:
#@title answer - copa
!lm_eval --model hf \
    --model_args pretrained=gpt2 \
    --tasks copa \
    --limit 50 \
    --device cuda:0


2026-04-27:00:01:56 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-27:00:02:09 INFO     [_cli.run:376] Selected Tasks: ['copa']
2026-04-27:00:02:14 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-27:00:02:14 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'gpt2'}
2026-04-27:00:02:22 INFO     [models.huggingface:161] Using device 'cuda:0'
config.json: 100% 665/665 [00:00<00:00, 2.03MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 89.7kB/s]
vocab.json: 1.04MB [00:00, 8.72MB/s]
merges.txt: 456kB [00:00, 7.16MB/s]
tokenizer.json: 1.36MB [00:00, 6.31MB/s]
2026-04-27:00:02:25 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
model.safetensors: 100% 548M/548M [00:13<00:00, 3